# LangGraph Single-Hop RAG

This notebook runs the real document QA pipeline through the first Agentic RAG graph. It parses the configured documents, creates embeddings, retrieves and reranks evidence, and calls the configured LLM.

The graph currently preserves the existing query path:

```text
START -> current_rag -> END
```

Later phases will replace the single node with routing, query rewriting, evidence grading, and verification nodes.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from within the project directory tree.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Markdown, display

from src.agent.graph import build_single_hop_rag_graph, invoke_single_hop_rag
from src.core.config import Config
from src.core.logger import setup_logging
from src.pipeline.query_runtime import build_query_pipeline

## Build the real RAG pipeline

This cell uses the same online query-runtime construction path as `main.py`. It loads existing chunks and document embeddings from Chroma, then rebuilds only the in-memory BM25 index. It does not parse documents or re-embed document chunks.

In [ ]:
config = Config()
setup_logging(config)
pipeline = build_query_pipeline(config)

checkpointer = InMemorySaver()
graph = build_single_hop_rag_graph(pipeline, checkpointer=checkpointer)

## Visualize the LangGraph workflow

`draw_mermaid()` exposes the compiled graph structure. LangSmith records executions of this graph when `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` are configured in `.env`.

In [ ]:
mermaid_graph = graph.get_graph().draw_mermaid()
display(Markdown(f'```mermaid\n{mermaid_graph}\n```'))

## Invoke the graph

`thread_id` identifies the checkpoint namespace. This invocation performs query embedding, hybrid retrieval, reranking, and LLM generation against the existing index. The current `InMemorySaver` is not durable across Python processes.

In [ ]:
thread_id = 'notebook-demo'
question = 'What classification accuracy did the ResNet26-V2 model achieve?'

response = invoke_single_hop_rag(
    graph,
    question,
    thread_id=thread_id,
)

display(Markdown(f'**Answer**: {response.answer}'))
for source in response.sources:
    print(f'- {source.chunk_id} | {source.source} | page {source.page}')

## Inspect checkpointed state

The checkpoint stores the state after graph execution. Future nodes will add rewritten queries, retrieved chunks, evidence decisions, verification status, and conversation messages to this state.

In [ ]:
graph_config = {'configurable': {'thread_id': thread_id}}
snapshot = graph.get_state(graph_config)
snapshot.values

## Check LangSmith configuration safely

This cell checks whether tracing is configured without printing the API key. When enabled, open the `doc-qa-agent` project in the LangSmith website to inspect the graph trace and its `current_rag` node.

In [ ]:
import os


langsmith_status = {
    'tracing_enabled': os.getenv('LANGSMITH_TRACING', '').lower() == 'true',
    'api_key_configured': bool(os.getenv('LANGSMITH_API_KEY')),
    'project': os.getenv('LANGSMITH_PROJECT', 'default'),
}
langsmith_status

## Runtime requirements

Build the index first with `python -m scripts.build_index` whenever source documents, parsing, chunking, filtering, or the embedding model changes. The online notebook requires `SCADS_API_KEY` for reranking and generation. It needs `MINERU_API_TOKEN` only when building the index. Configure `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, and `LANGSMITH_PROJECT=doc-qa-agent` to view traces in LangSmith. Do not run indexing against sensitive documents unless uploading their content to MinerU and ScaDS.AI is permitted.